# Sistema RAG-SQL: Generación Automática de Consultas SQL con LLM

Este notebook demuestra un sistema RAG completo que:
1. **Recibe preguntas en lenguaje natural**
2. **Genera consultas SQL automáticamente** usando Gemini
3. **Ejecuta las consultas** en la base de datos
4. **Genera respuestas en lenguaje natural** basadas en los resultados

## Arquitectura del Sistema
- **LLM**: Google Gemini Flash para generación de SQL y respuestas
- **Embeddings**: Text-Embedding-Gecko (opcional para búsqueda vectorial)
- **Base de datos**: SQLite/PostgreSQL intercambiable
- **Framework**: LangChain + LangGraph para orquestación

## 1. Configuración del Entorno

In [ ]:
# Instalación de dependencias
!pip install -q langchain langchain-google-genai langchain-community sqlalchemy pandas python-dotenv

In [ ]:
import os
import getpass
from dotenv import load_dotenv
import pandas as pd
import sqlite3
from typing_extensions import TypedDict, Annotated

# Configuración de API keys
load_dotenv()

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Ingrese su Google API Key: ")

print("✅ Configuración completada")

## 2. Inicialización del LLM y Herramientas

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

# Inicializar LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    convert_system_message_to_human=True
)

# Inicializar embeddings (opcional)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004"
)

print("✅ LLM y embeddings inicializados")

## 3. Creación de Base de Datos de Ejemplo

In [ ]:
# Crear base de datos de ejemplo
def crear_base_datos_ejemplo():
    conn = sqlite3.connect('ejemplo_universidad.db')
    cursor = conn.cursor()
    
    # Tabla Estudiantes
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS estudiantes (
        id INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        apellido TEXT NOT NULL,
        edad INTEGER,
        carrera TEXT,
        promedio REAL,
        fecha_ingreso DATE
    )
    ''')
    
    # Tabla Cursos
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS cursos (
        id INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        creditos INTEGER,
        profesor TEXT,
        departamento TEXT
    )
    ''')
    
    # Tabla Inscripciones
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS inscripciones (
        id INTEGER PRIMARY KEY,
        estudiante_id INTEGER,
        curso_id INTEGER,
        calificacion REAL,
        semestre TEXT,
        FOREIGN KEY (estudiante_id) REFERENCES estudiantes (id),
        FOREIGN KEY (curso_id) REFERENCES cursos (id)
    )
    ''')
    
    # Datos de ejemplo
    estudiantes_data = [
        (1, 'Juan', 'Pérez', 20, 'Ingeniería Informática', 8.5, '2022-03-01'),
        (2, 'María', 'González', 21, 'Ingeniería Informática', 9.2, '2021-03-01'),
        (3, 'Carlos', 'Rodríguez', 22, 'Ingeniería Industrial', 7.8, '2020-03-01'),
        (4, 'Ana', 'López', 19, 'Matemáticas', 9.5, '2023-03-01'),
        (5, 'Luis', 'Martín', 23, 'Física', 8.1, '2019-03-01')
    ]
    
    cursos_data = [
        (1, 'Algoritmos y Estructuras de Datos', 6, 'Dr. Smith', 'Informática'),
        (2, 'Cálculo I', 8, 'Dra. Johnson', 'Matemáticas'),
        (3, 'Física General', 6, 'Dr. Brown', 'Física'),
        (4, 'Base de Datos', 6, 'Dr. Davis', 'Informática'),
        (5, 'Estadística', 4, 'Dra. Wilson', 'Matemáticas')
    ]
    
    inscripciones_data = [
        (1, 1, 1, 8.5, '2023-1'),
        (2, 1, 2, 7.8, '2023-1'),
        (3, 2, 1, 9.2, '2023-1'),
        (4, 2, 4, 9.0, '2023-1'),
        (5, 3, 3, 7.5, '2023-1'),
        (6, 4, 2, 9.8, '2023-1'),
        (7, 5, 3, 8.2, '2023-1')
    ]
    
    cursor.executemany('INSERT OR REPLACE INTO estudiantes VALUES (?,?,?,?,?,?,?)', estudiantes_data)
    cursor.executemany('INSERT OR REPLACE INTO cursos VALUES (?,?,?,?,?)', cursos_data)
    cursor.executemany('INSERT OR REPLACE INTO inscripciones VALUES (?,?,?,?,?)', inscripciones_data)
    
    conn.commit()
    conn.close()
    print("✅ Base de datos creada exitosamente")

crear_base_datos_ejemplo()

# Conectar con LangChain
db = SQLDatabase.from_uri("sqlite:///ejemplo_universidad.db")
print(f"Conectado a base de datos: {db.dialect}")
print(f"Tablas disponibles: {db.get_usable_table_names()}")

## 4. Definir el Estado del Sistema RAG-SQL

In [ ]:
# Estado del sistema RAG-SQL
class EstadoRAGSQL(TypedDict):
    pregunta: str               # Pregunta original del usuario
    pregunta_expandida: str     # Pregunta expandida/mejorada
    consulta_sql: str          # Consulta SQL generada
    resultado_sql: str         # Resultado de la consulta
    respuesta: str             # Respuesta final en lenguaje natural
    contexto_esquema: str      # Información del esquema de la DB

print("✅ Estado del sistema definido")

## 5. Expansión de Consultas (Query Expansion)

In [ ]:
def expandir_consulta(estado: EstadoRAGSQL) -> dict:
    """
    Expande y mejora la consulta del usuario para obtener mejores resultados SQL
    """
    prompt_expansion = ChatPromptTemplate.from_messages([
        ("system", """
        Eres un experto en análisis de consultas de bases de datos. Tu tarea es expandir y mejorar 
        la pregunta del usuario para que sea más específica y completa, considerando el contexto 
        de una base de datos universitaria.
        
        Contexto de la base de datos:
        - Tabla 'estudiantes': información personal y académica de estudiantes
        - Tabla 'cursos': información sobre materias y profesores
        - Tabla 'inscripciones': relación entre estudiantes y cursos con calificaciones
        
        Expande la pregunta agregando contexto relevante pero manteniendo la intención original.
        Si la pregunta ya es específica, mantenla igual.
        """),
        ("human", "Pregunta original: {pregunta}\n\nPregunta expandida:")
    ])
    
    chain = prompt_expansion | llm
    resultado = chain.invoke({"pregunta": estado["pregunta"]})
    
    return {"pregunta_expandida": resultado.content}

# Prueba de expansión
estado_test = {"pregunta": "¿Cuántos estudiantes hay?"}
resultado_expansion = expandir_consulta(estado_test)
print(f"Pregunta original: {estado_test['pregunta']}")
print(f"Pregunta expandida: {resultado_expansion['pregunta_expandida']}")

## 6. Generación Automática de SQL

In [ ]:
# Definir estructura para la salida SQL
class SalidaSQL(TypedDict):
    """Consulta SQL generada."""
    consulta: Annotated[str, "", "Consulta SQL sintácticamente válida."]

def generar_sql(estado: EstadoRAGSQL) -> dict:
    """
    Genera consulta SQL automáticamente a partir de la pregunta en lenguaje natural
    """
    # Obtener información del esquema
    schema_info = db.get_table_info()
    
    # Prompt para generación de SQL siguiendo mejores prácticas de LangChain
    prompt_sql = ChatPromptTemplate.from_messages([
        ("system", """
        Dado una pregunta de entrada, crea una consulta {dialect} sintácticamente correcta para ejecutar.
        A menos que el usuario especifique en su pregunta un número específico de ejemplos que desea obtener,
        siempre limita tu consulta a un máximo de {top_k} resultados.
        
        Puedes ordenar los resultados por una columna relevante para devolver los ejemplos más interesantes.
        
        Nunca consultes todas las columnas de una tabla específica, solo solicita las pocas columnas 
        relevantes dada la pregunta.
        
        Presta atención a usar solo los nombres de columnas que puedes ver en la descripción del esquema.
        Ten cuidado de no consultar columnas que no existen. También, presta atención a qué columna 
        está en qué tabla.
        
        Solo usa las siguientes tablas:
        {table_info}
        
        IMPORTANTE: Responde SOLO con la consulta SQL, sin explicaciones adicionales.
        """),
        ("human", "Pregunta: {pregunta}")
    ])
    
    # Usar structured output para obtener SQL limpio
    llm_estructurado = llm.with_structured_output(SalidaSQL)
    
    # Preparar el prompt con información del esquema
    prompt_formateado = prompt_sql.invoke({
        "dialect": db.dialect,
        "top_k": 10,
        "table_info": schema_info,
        "pregunta": estado.get("pregunta_expandida", estado["pregunta"])
    })
    
    resultado = llm_estructurado.invoke(prompt_formateado)
    
    return {
        "consulta_sql": resultado["consulta"],
        "contexto_esquema": schema_info
    }

# Prueba de generación SQL
estado_test = {
    "pregunta": "¿Cuántos estudiantes hay?",
    "pregunta_expandida": "¿Cuántos estudiantes hay registrados en total en la base de datos?"
}
resultado_sql = generar_sql(estado_test)
print(f"SQL generado: {resultado_sql['consulta_sql']}")

## 7. Ejecución de Consultas SQL

In [ ]:
from langchain_community.tools.sql_database.tool import QuerySQLDatabaseTool

def ejecutar_sql(estado: EstadoRAGSQL) -> dict:
    """
    Ejecuta la consulta SQL generada en la base de datos
    """
    # Crear herramienta de ejecución
    ejecutor_sql = QuerySQLDatabaseTool(db=db)
    
    try:
        resultado = ejecutor_sql.invoke(estado["consulta_sql"])
        return {"resultado_sql": resultado}
    except Exception as e:
        return {"resultado_sql": f"Error al ejecutar SQL: {str(e)}"}

# Prueba de ejecución
estado_test = {
    "consulta_sql": "SELECT COUNT(*) as total_estudiantes FROM estudiantes;"
}
resultado_ejecucion = ejecutar_sql(estado_test)
print(f"Resultado: {resultado_ejecucion['resultado_sql']}")

## 8. Generación de Respuesta en Lenguaje Natural

In [ ]:
def generar_respuesta(estado: EstadoRAGSQL) -> dict:
    """
    Genera respuesta en lenguaje natural basada en los resultados de la consulta SQL
    """
    prompt_respuesta = ChatPromptTemplate.from_messages([
        ("system", """
        Eres un asistente experto que ayuda a interpretar resultados de bases de datos.
        Dada una pregunta del usuario, la consulta SQL ejecutada y su resultado,
        proporciona una respuesta clara y útil en lenguaje natural.
        
        INSTRUCCIONES:
        1. Responde de manera conversacional y amigable
        2. Incluye los datos más relevantes del resultado
        3. Si hay múltiples resultados, presenta un resumen claro
        4. Si no hay resultados, explica por qué podría ser
        5. Mantén la respuesta concisa pero informativa
        """),
        ("human", """
        Pregunta del usuario: {pregunta}
        Consulta SQL ejecutada: {consulta_sql}
        Resultado SQL: {resultado_sql}
        
        Respuesta en lenguaje natural:
        """)
    ])
    
    chain = prompt_respuesta | llm
    resultado = chain.invoke({
        "pregunta": estado["pregunta"],
        "consulta_sql": estado["consulta_sql"],
        "resultado_sql": estado["resultado_sql"]
    })
    
    return {"respuesta": resultado.content}

# Prueba de generación de respuesta
estado_test = {
    "pregunta": "¿Cuántos estudiantes hay?",
    "consulta_sql": "SELECT COUNT(*) as total_estudiantes FROM estudiantes;",
    "resultado_sql": "[(5,)]"
}
resultado_respuesta = generar_respuesta(estado_test)
print(f"Respuesta: {resultado_respuesta['respuesta']}")

## 9. Orquestación con LangGraph

In [ ]:
from langgraph.graph import START, StateGraph

# Crear el grafo del pipeline RAG-SQL
constructor_grafo = StateGraph(EstadoRAGSQL)

# Agregar nodos al grafo
constructor_grafo.add_node("expandir_consulta", expandir_consulta)
constructor_grafo.add_node("generar_sql", generar_sql)
constructor_grafo.add_node("ejecutar_sql", ejecutar_sql)
constructor_grafo.add_node("generar_respuesta", generar_respuesta)

# Definir el flujo del pipeline
constructor_grafo.add_edge(START, "expandir_consulta")
constructor_grafo.add_edge("expandir_consulta", "generar_sql")
constructor_grafo.add_edge("generar_sql", "ejecutar_sql")
constructor_grafo.add_edge("ejecutar_sql", "generar_respuesta")

# Compilar el grafo
grafo_rag_sql = constructor_grafo.compile()

print("✅ Pipeline RAG-SQL configurado exitosamente")

## 10. Función Principal del Sistema RAG-SQL

In [ ]:
def consultar_rag_sql(pregunta: str, mostrar_pasos: bool = True) -> str:
    """
    Función principal que procesa una pregunta usando el pipeline RAG-SQL completo
    
    Args:
        pregunta: Pregunta en lenguaje natural del usuario
        mostrar_pasos: Si mostrar los pasos intermedios del proceso
    
    Returns:
        Respuesta en lenguaje natural
    """
    print(f"🤔 Procesando pregunta: '{pregunta}'\n")
    
    # Ejecutar el pipeline completo
    resultado_completo = None
    
    for paso in grafo_rag_sql.stream({"pregunta": pregunta}, stream_mode="updates"):
        for nombre_nodo, datos in paso.items():
            if mostrar_pasos:
                if nombre_nodo == "expandir_consulta":
                    print(f"🔍 **Expansión de consulta:**")
                    print(f"   Pregunta expandida: {datos['pregunta_expandida']}\n")
                
                elif nombre_nodo == "generar_sql":
                    print(f"💾 **Generación SQL:**")
                    print(f"   ```sql\n   {datos['consulta_sql']}\n   ```\n")
                
                elif nombre_nodo == "ejecutar_sql":
                    print(f"⚡ **Ejecución SQL:**")
                    print(f"   Resultado: {datos['resultado_sql']}\n")
                
                elif nombre_nodo == "generar_respuesta":
                    print(f"💬 **Respuesta final:**")
                    print(f"   {datos['respuesta']}\n")
                    resultado_completo = datos['respuesta']
    
    return resultado_completo

print("✅ Sistema RAG-SQL listo para usar")

## 11. Pruebas del Sistema RAG-SQL

In [ ]:
# Prueba 1: Consulta simple de conteo
print("=" * 80)
print("PRUEBA 1: Consulta simple")
print("=" * 80)
respuesta1 = consultar_rag_sql("¿Cuántos estudiantes hay en total?")

In [ ]:
# Prueba 2: Consulta con filtros
print("=" * 80)
print("PRUEBA 2: Consulta con filtros")
print("=" * 80)
respuesta2 = consultar_rag_sql("¿Qué estudiantes tienen un promedio mayor a 9.0?")

In [ ]:
# Prueba 3: Consulta con JOIN
print("=" * 80)
print("PRUEBA 3: Consulta con JOIN")
print("=" * 80)
respuesta3 = consultar_rag_sql("¿Qué estudiantes están inscritos en el curso de Algoritmos?")

In [ ]:
# Prueba 4: Consulta analítica
print("=" * 80)
print("PRUEBA 4: Consulta analítica")
print("=" * 80)
respuesta4 = consultar_rag_sql("¿Cuál es el promedio de calificaciones por carrera?")

In [ ]:
# Prueba 5: Consulta compleja
print("=" * 80)
print("PRUEBA 5: Consulta compleja")
print("=" * 80)
respuesta5 = consultar_rag_sql("¿Qué profesor tiene estudiantes con las mejores calificaciones?")

## 12. Sistema RAG Documental (Opcional)

In [ ]:
# Sistema RAG híbrido: SQL + Documental
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.text_splitter import RecursiveCharacterTextSplitter

def crear_rag_documental():
    """
    Crea un sistema RAG documental complementario usando embeddings
    """
    # Documentos de ejemplo sobre la universidad
    documentos = [
        "La Universidad ofrece carreras en Ingeniería Informática, Ingeniería Industrial, Matemáticas y Física.",
        "Los estudiantes pueden inscribirse en múltiples cursos por semestre.",
        "El sistema de calificaciones va de 0 a 10, donde 6 es la nota mínima para aprobar.",
        "Los profesores pertenecen a diferentes departamentos: Informática, Matemáticas y Física.",
        "La universidad tiene políticas estrictas sobre el rendimiento académico y el promedio mínimo."
    ]
    
    # Crear vector store
    vector_store = InMemoryVectorStore(embeddings)
    vector_store.add_texts(documentos)
    
    return vector_store

def consulta_hibrida(pregunta: str, usar_documentos: bool = True):
    """
    Combina RAG-SQL con búsqueda documental para respuestas más completas
    """
    print(f"🔄 Consulta híbrida: {pregunta}\n")
    
    # 1. Ejecutar RAG-SQL
    respuesta_sql = consultar_rag_sql(pregunta, mostrar_pasos=False)
    
    if usar_documentos:
        # 2. Buscar en documentos
        vector_store = crear_rag_documental()
        docs_relevantes = vector_store.similarity_search(pregunta, k=2)
        contexto_documental = "\n".join([doc.page_content for doc in docs_relevantes])
        
        # 3. Combinar respuestas
        prompt_hibrido = ChatPromptTemplate.from_messages([
            ("system", """
            Combina la información de la consulta SQL con el contexto documental para dar 
            una respuesta más completa y contextualizada.
            """),
            ("human", """
            Pregunta: {pregunta}
            
            Respuesta basada en datos SQL: {respuesta_sql}
            
            Contexto documental adicional: {contexto_documental}
            
            Respuesta combinada:
            """)
        ])
        
        chain = prompt_hibrido | llm
        respuesta_final = chain.invoke({
            "pregunta": pregunta,
            "respuesta_sql": respuesta_sql,
            "contexto_documental": contexto_documental
        })
        
        print(f"📊 **Respuesta combinada (SQL + Documentos):**")
        print(respuesta_final.content)
        return respuesta_final.content
    else:
        return respuesta_sql

# Prueba del sistema híbrido
print("=" * 80)
print("PRUEBA SISTEMA HÍBRIDO")
print("=" * 80)
consulta_hibrida("¿Cómo está el rendimiento académico de los estudiantes?")

## 13. Interfaz de Usuario Interactiva

In [ ]:
def interfaz_interactiva():
    """
    Interfaz simple para interactuar con el sistema RAG-SQL
    """
    print("\n" + "="*60)
    print("🎓 SISTEMA RAG-SQL UNIVERSITARIO")
    print("="*60)
    print("Pregunta sobre estudiantes, cursos o inscripciones.")
    print("Escribe 'salir' para terminar.\n")
    
    ejemplos = [
        "¿Cuántos estudiantes hay por carrera?",
        "¿Qué estudiante tiene el mejor promedio?",
        "¿Cuáles son los cursos más populares?",
        "¿Qué profesores enseñan en el departamento de Informática?"
    ]
    
    print("💡 **Ejemplos de preguntas:**")
    for i, ejemplo in enumerate(ejemplos, 1):
        print(f"   {i}. {ejemplo}")
    print()
    
    while True:
        try:
            pregunta = input("🤔 Tu pregunta: ").strip()
            
            if pregunta.lower() in ['salir', 'exit', 'quit']:
                print("\n👋 ¡Hasta luego!")
                break
            
            if not pregunta:
                print("⚠️  Por favor, ingresa una pregunta.\n")
                continue
            
            print("\n" + "-"*50)
            respuesta = consultar_rag_sql(pregunta)
            print("-"*50 + "\n")
            
        except KeyboardInterrupt:
            print("\n\n👋 ¡Hasta luego!")
            break
        except Exception as e:
            print(f"❌ Error: {str(e)}\n")

# Descomenta la siguiente línea para ejecutar la interfaz interactiva
# interfaz_interactiva()

## 14. Análisis de Rendimiento y Métricas

In [ ]:
import time
from datetime import datetime

def evaluar_rendimiento():
    """
    Evalúa el rendimiento del sistema RAG-SQL con diferentes tipos de consultas
    """
    preguntas_test = [
        "¿Cuántos estudiantes hay?",
        "¿Cuál es el promedio general de calificaciones?",
        "¿Qué estudiantes están en Ingeniería Informática?",
        "¿Cuáles son los cursos con más créditos?",
        "¿Qué profesor tiene más estudiantes inscritos?"
    ]
    
    resultados = []
    
    print("📊 **EVALUACIÓN DE RENDIMIENTO**\n")
    
    for i, pregunta in enumerate(preguntas_test, 1):
        print(f"Prueba {i}/5: {pregunta}")
        
        inicio = time.time()
        try:
            respuesta = consultar_rag_sql(pregunta, mostrar_pasos=False)
            tiempo_total = time.time() - inicio
            estado = "✅ Éxito"
        except Exception as e:
            tiempo_total = time.time() - inicio
            estado = f"❌ Error: {str(e)[:50]}..."
            respuesta = None
        
        resultados.append({
            'pregunta': pregunta,
            'tiempo': tiempo_total,
            'estado': estado,
            'respuesta_exitosa': respuesta is not None
        })
        
        print(f"   Tiempo: {tiempo_total:.2f}s - {estado}\n")
    
    # Resumen de métricas
    tiempo_promedio = sum(r['tiempo'] for r in resultados) / len(resultados)
    tasa_exito = sum(r['respuesta_exitosa'] for r in resultados) / len(resultados) * 100
    tiempo_max = max(r['tiempo'] for r in resultados)
    tiempo_min = min(r['tiempo'] for r in resultados)
    
    print("="*60)
    print("📈 **MÉTRICAS DE RENDIMIENTO**")
    print("="*60)
    print(f"⏱️  Tiempo promedio: {tiempo_promedio:.2f} segundos")
    print(f"✅ Tasa de éxito: {tasa_exito:.1f}%")
    print(f"🚀 Consulta más rápida: {tiempo_min:.2f}s")
    print(f"🐌 Consulta más lenta: {tiempo_max:.2f}s")
    
    return resultados

# Ejecutar evaluación de rendimiento
metricas = evaluar_rendimiento()

## 15. Configuración para Producción

In [ ]:
# Configuración avanzada para entorno de producción
def configuracion_produccion():
    """
    Configuraciones recomendadas para un entorno de producción
    """
    config = {
        "database": {
            "type": "postgresql",  # Cambiar a PostgreSQL en producción
            "connection_pool_size": 20,
            "max_overflow": 30,
            "pool_timeout": 30,
            "ssl_mode": "require"
        },
        "llm": {
            "model": "gemini-1.5-pro",  # Modelo más potente para producción
            "temperature": 0,
            "max_tokens": 1000,
            "timeout": 30,
            "retry_attempts": 3
        },
        "seguridad": {
            "query_validation": True,
            "sql_injection_protection": True,
            "allowed_operations": ["SELECT"],  # Solo consultas de lectura
            "max_query_complexity": 10,
            "rate_limiting": {
                "requests_per_minute": 60,
                "requests_per_hour": 1000
            }
        },
        "monitoring": {
            "enable_logging": True,
            "log_level": "INFO",
            "enable_metrics": True,
            "enable_tracing": True
        },
        "cache": {
            "enable_query_cache": True,
            "cache_ttl": 3600,  # 1 hora
            "max_cache_size": 1000
        }
    }
    
    print("🔧 **CONFIGURACIÓN DE PRODUCCIÓN**")
    print("="*50)
    
    for categoria, configuraciones in config.items():
        print(f"\n📋 **{categoria.upper()}:**")
        if isinstance(configuraciones, dict):
            for key, value in configuraciones.items():
                print(f"   • {key}: {value}")
        else:
            print(f"   • {configuraciones}")
    
    return config

config_prod = configuracion_produccion()

## 16. Integración con Otras Bibliotecas

In [ ]:
# Código de integración con VannaAI y PandasAI (requiere instalación adicional)

def integracion_vanna():
    """
    Ejemplo de integración con VannaAI para generación de SQL especializada
    """
    codigo_ejemplo = '''
# Instalar: pip install vanna
import vanna as vn

# Configurar VannaAI
vn.set_api_key("tu-api-key")
vn.set_model("chinook")  # O tu modelo personalizado

def generar_sql_vanna(pregunta: str) -> str:
    """Generar SQL usando VannaAI"""
    sql = vn.generate_sql(pregunta)
    return sql

def consulta_con_vanna(pregunta: str):
    """Pipeline completo con VannaAI"""
    sql = generar_sql_vanna(pregunta)
    resultado = db.run(sql)
    respuesta = vn.generate_explanation(sql, resultado)
    return respuesta
    '''
    
    print("🔧 **INTEGRACIÓN CON VANNA AI**")
    print("=" * 40)
    print(codigo_ejemplo)

def integracion_pandas_ai():
    """
    Ejemplo de integración con PandasAI para análisis de datos
    """
    codigo_ejemplo = '''
# Instalar: pip install pandasai
from pandasai import SmartDataframe
from pandasai.llm import GoogleGemini

def consulta_con_pandas_ai(pregunta: str):
    """Análisis con PandasAI"""
    # Obtener datos como DataFrame
    df = pd.read_sql_query("SELECT * FROM estudiantes", db._engine)
    
    # Configurar LLM
    llm = GoogleGemini(api_token="tu-api-key")
    
    # Crear SmartDataframe
    smart_df = SmartDataframe(df, config={"llm": llm})
    
    # Hacer consulta en lenguaje natural
    resultado = smart_df.chat(pregunta)
    return resultado
    '''
    
    print("🔧 **INTEGRACIÓN CON PANDAS AI**")
    print("=" * 40)
    print(codigo_ejemplo)

# Mostrar ejemplos de integración
integracion_vanna()
print("\n")
integracion_pandas_ai()

## 17. Resumen y Conclusiones

In [ ]:
def resumen_sistema():
    """
    Resumen completo del sistema RAG-SQL implementado
    """
    print("🎯 **RESUMEN DEL SISTEMA RAG-SQL**")
    print("=" * 60)
    
    componentes = {
        "🧠 LLM Principal": "Google Gemini 1.5 Flash",
        "📊 Base de Datos": "SQLite (intercambiable a PostgreSQL)",
        "🔍 Embeddings": "Google Text-Embedding-004",
        "⚡ Framework": "LangChain + LangGraph",
        "🔄 Pipeline": "Expansión → SQL → Ejecución → Respuesta"
    }
    
    funcionalidades = [
        "✅ Generación automática de SQL desde lenguaje natural",
        "✅ Expansión inteligente de consultas",
        "✅ Ejecución segura de consultas",
        "✅ Respuestas en lenguaje natural",
        "✅ Sistema híbrido (SQL + Documental)",
        "✅ Interfaz interactiva",
        "✅ Métricas de rendimiento",
        "✅ Configuración para producción",
        "✅ Integración con VannaAI y PandasAI"
    ]
    
    casos_uso = [
        "📈 Análisis de datos empresariales",
        "🎓 Sistemas universitarios",
        "💼 Business Intelligence",
        "📊 Dashboards interactivos",
        "🤖 Asistentes de datos"
    ]
    
    print("\n🏗️ **COMPONENTES PRINCIPALES:**")
    for componente, descripcion in componentes.items():
        print(f"   {componente}: {descripcion}")
    
    print("\n⚙️ **FUNCIONALIDADES IMPLEMENTADAS:**")
    for func in funcionalidades:
        print(f"   {func}")
    
    print("\n🎯 **CASOS DE USO:**")
    for caso in casos_uso:
        print(f"   {caso}")
    
    print("\n🔮 **PRÓXIMOS PASOS:**")
    print("   • Implementar validación avanzada de SQL")
    print("   • Agregar cache inteligente")
    print("   • Implementar autenticación y autorización")
    print("   • Crear API REST para integración externa")
    print("   • Desarrollar dashboard web interactivo")
    
    print("\n" + "=" * 60)
    print("🚀 **¡SISTEMA RAG-SQL COMPLETAMENTE FUNCIONAL!** 🚀")
    print("=" * 60)

resumen_sistema()